# 参数化 Bootstrap（对数正态）

> 对应代码：`MathStatsCode/code_in_notes/Chap.15/bootstrap_lognormal.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：从小样本出发（σ² 已知）做参数化 Bootstrap 再抽样，并计算均值参数的偏差修正估计。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.15`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.15")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear all`、`set more off`、`set seed 1000`，准备模拟环境。本 notebook 演示**参数化 Bootstrap**：估计对数正态分布均值参数 μ（已知对数方差 σ²=2）。


In [ ]:
%%stata
clear all
set more off
set seed 1000
set obs 10


**产生样本**

生成原始样本：`local mu=2`、`gen z=rnormal(mu,sqrt(2))`（对数服从 N(2,2)）、`gen x=exp(z)`（对数正态变量），样本量 n=10（`set obs 10`）。


In [ ]:
%%stata
local mu=2
gen z=rnormal(`mu',sqrt(2))
gen x=exp(z)


**进行估计**

计算点估计：`su x` 得到样本均值后，`local mu_hat=log(r(mean))-1`——由对数正态的均值公式 E[x]=exp(μ+σ²/2)=exp(μ+1) 反解出 μ 的估计。


In [ ]:
%%stata
su x
local mu_hat=log(r(mean))-1
di `mu_hat'


**进行再抽样，将再抽样结果放入到数据框bootstrap中**

**参数化再抽样循环**（核心）：`frame create bootstrap mu_star` 新建结果框；设定 `B=10000` 次；每次循环中 `gen z_star=rnormal(mu_hat, sqrt(2))` 用估计出的 μ 结合已知 σ² 重新生成样本——这就是“参数化”Bootstrap（把估计值当作真值）；`gen x_star=exp(z_star)` 得到 bootstrap 样本，求其均值并用同一公式算出 `mu_star`，`frame post` 写入结果框，最后删掉临时变量进入下一轮。之后画 mu_star 的直方图、导出，并计算 **偏差修正估计** `mu_bc = 2×mu_hat - mu_star_bar`（偏差修正的经典公式：原始估计减去 bootstrap 估计的平均偏差）。


In [ ]:
%%stata
frame create bootstrap mu_star
local B=10000
forvalues i=1/`B'{
	quietly{
		gen z_star=rnormal(`mu_hat',sqrt(2))
		gen x_star=exp(z_star)
		su x_star
		local mu_star=log(r(mean))-1
		frame post bootstrap (`mu_star')
		drop z_star
		drop x_star
	}
}
frame bootstrap: hist mu_star
frame bootstrap: su mu_star, de
local mu_star_bar=r(mean)
di "mu_bc=" _skip 2*`mu_hat'-`mu_star_bar'
